In [ ]:
import json
import pandas as pd
import numpy as np
import re
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

In [ ]:
import json
import pandas as pd
import re

def load_robustly(file_path):
    records = []
    malformed_objects_count = 0
    try:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()

        # Strip outer brackets if it's a single JSON array
        cleaned_content = content.strip()
        if cleaned_content.startswith('[') and cleaned_content.endswith(']') and len(cleaned_content) > 2:
            cleaned_content = cleaned_content[1:-1]

        # Replace '},{' with '}\n{' to get each object on a new line for easier parsing
        object_strings = re.split(r'}\s*,\s*\{', cleaned_content)

        # Reconstruct full object strings with their braces
        parsed_object_strings = []
        if len(object_strings) == 1 and object_strings[0].strip(): # Single object case
            parsed_object_strings.append(object_strings[0].strip())
        elif len(object_strings) > 1: # Multiple objects
            parsed_object_strings.append(object_strings[0].strip() + '}')
            for i in range(1, len(object_strings) - 1):
                parsed_object_strings.append('{' + object_strings[i].strip() + '}')
            parsed_object_strings.append('{' + object_strings[-1].strip())

        for i, obj_str in enumerate(parsed_object_strings):
            obj_str = obj_str.strip()
            if not obj_str:
                continue
            # Ensure the string is wrapped in braces before attempting to parse
            if not obj_str.startswith('{'):
                obj_str = '{' + obj_str
            if not obj_str.endswith('}'):
                obj_str = obj_str + '}'

            try:
                records.append(json.loads(obj_str))
            except json.JSONDecodeError as e:
                malformed_objects_count += 1
                # print(f"Skipped malformed object (index {i}) in {file_path}. Error: {e}. Content start: {obj_str[:100]}...")
            except Exception as e:
                malformed_objects_count += 1
                # print(f"Skipped object (index {i}) due to unexpected error in {file_path}. Error: {e}. Content start: {obj_str[:100]}...")

    except Exception as e:
        print(f"An unexpected error occurred while reading or processing {file_path}: {e}")
        return [] # Return empty list on major file processing errors

    print(f"Successfully loaded {len(records)} records from {file_path}.")
    print(f"Skipped {malformed_objects_count} malformed objects from {file_path}.")
    return records

cleaned_data = load_robustly(
    "/content/drive/MyDrive/Infosys_Datasets/cleaned_enron_emails.json"
)
threaded_data = load_robustly(
    "/content/drive/MyDrive/Infosys_Datasets/threaded_emails.json"
)

cleaned_df = pd.DataFrame(cleaned_data)
threaded_df = pd.DataFrame(threaded_data)

print(f"cleaned_df shape: {cleaned_df.shape}")
print(f"threaded_df shape: {threaded_df.shape}")

Successfully loaded 482552 records from /content/drive/MyDrive/Infosys_Datasets/cleaned_enron_emails.json.
Skipped 8 malformed objects from /content/drive/MyDrive/Infosys_Datasets/cleaned_enron_emails.json.
Successfully loaded 22974 records from /content/drive/MyDrive/Infosys_Datasets/threaded_emails.json.
Skipped 26444 malformed objects from /content/drive/MyDrive/Infosys_Datasets/threaded_emails.json.
cleaned_df shape: (482552, 7)
threaded_df shape: (22974, 11)


In [ ]:
cleaned_df.head()
threaded_df.head()

,MessageID,From,To,InReplyTo,Subject,Date,Body,ThreadKey,Filename,ThreadID,ThreadPosition
0,<23987417.1075857585124.JavaMail.evans@thyme>,slafontaine@globalp.com,john.arnold@enron.com,,re:summer inverses,"Thu, 07 Dec 2000 01:27:00 -0800",they are crazy but mite have to scale in which...,"summer inverses::Thu, 07 Dec 2000 01:27:00 -0800",23.,thread-218b74fb-b061-43b7-a3a0-3935a88ec145,3
1,<19171686.1075857585034.JavaMail.evans@thyme>,slafontaine@globalp.com,john.arnold@enron.com,,re:summer inverses,"Fri, 08 Dec 2000 05:05:00 -0800",i suck-hope youve made more money in natgas la...,"summer inverses::Fri, 08 Dec 2000 05:05:00 -0800",19.,thread-218b74fb-b061-43b7-a3a0-3935a88ec145,6
2,<3552781.1075857584209.JavaMail.evans@thyme>,john.arnold@enron.com,slafontaine@globalp.com,,re:summer inverses,"Mon, 11 Dec 2000 08:51:00 -0800",amazing how with cash futures at $1 and the ba...,"summer inverses::Mon, 11 Dec 2000 08:51:00 -0800",187.,thread-218b74fb-b061-43b7-a3a0-3935a88ec145,9
3,<30245340.1075852710341.JavaMail.evans@thyme>,john.arnold@enron.com,steve.lafontaine@bankofamerica.com,,Re:,"Sun, 13 May 2001 20:33:00 -0700",most bullish thing at this point is moving clo...,"::Sun, 13 May 2001 20:33:00 -0700",4.,thread-7073203c-410c-43da-a3bc-551e58f31f9f,3
4,<1864054.1075857631234.JavaMail.evans@thyme>,steve.lafontaine@bankofamerica.com,john.arnold@enron.com,,RE:,"Mon, 14 May 2001 00:51:00 -0700",i think thats rite -think curve flattens somew...,"::Mon, 14 May 2001 00:51:00 -0700",69.,thread-7073203c-410c-43da-a3bc-551e58f31f9f,4


In [ ]:
!sed -n '971300,971330p' /content/drive/MyDrive/Infosys_Datasets/cleaned_enron_emails.json

  {
    "From": "twanda.sweet@enron.com",
    "To": "aleck.dadson@enron.com",
    "Subject": "Project Stanley",
    "Date": "Mon, 12 Jun 2000 10:50:00 -0700",
    "Body": "Mr. Dadson, please be advised that a conference call regarding the\nabove-referenced matter has been scheduled for this Thursday at 9:00am\ncentral standard time.  The participants are as follows:\n\nRichard Sanders\nRob Hemstock (Calgary)\nRick Shapiro\nAl Dadson\n\nPlease let me know if this time is not convenient for you.  Also, please\nprovide me with a phone number.  The number that we currently have in the\nEnron directory is not a working number (416-214-1740).\n\nThanks\nTwanda\n713-853-9402",
    "ThreadKey": "project stanley::Mon, 12 Jun 2000 10:50:00 -0700",
    "Filename": "175."
  },
  {
    "From": "richard.sanders@enron.com",
    "To": "robert.williams@enron.com",
    "Subject": "Re: FW: Strategy & End Game for FERC Proceeding",
    "Date": "Mon, 08 Jan 2001 02:32:00 -0800",
    "Body": "This is an ENA

In [ ]:
import json
import pandas as pd

def load_json_safely(file_path):
    records = []
    bad_lines = 0

    with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f, start=1):
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                bad_lines += 1

    print(f"Loaded records: {len(records)}")
    print(f"Skipped corrupted lines: {bad_lines}")
    return records

cleaned_data = load_json_safely(
    "/content/drive/MyDrive/Infosys_Datasets/cleaned_enron_emails.json"
)

threaded_data = load_json_safely(
    "/content/drive/MyDrive/Infosys_Datasets/threaded_emails.json"
)

cleaned_df = pd.DataFrame(cleaned_data)
threaded_df = pd.DataFrame(threaded_data)

Loaded records: 0
Skipped corrupted lines: 4343002
Loaded records: 0
Skipped corrupted lines: 3444622


In [ ]:
import pandas as pd
import json

try:
    cleaned_df = pd.read_json(
        "/content/drive/MyDrive/Infosys_Datasets/cleaned_enron_emails.json",
        encoding="utf-8",
        encoding_errors="ignore"
    )
except (ValueError, json.JSONDecodeError) as e:
    print(f"Error loading cleaned_enron_emails.json: {e}")
    cleaned_df = pd.DataFrame() # Initialize as empty DataFrame on error

try:
    threaded_df = pd.read_json(
        "/content/drive/MyDrive/Infosys_Datasets/threaded_emails.json",
        encoding="utf-8",
        encoding_errors="ignore"
    )
except (ValueError, json.JSONDecodeError) as e:
    print(f"Error loading threaded_emails.json: {e}")
    threaded_df = pd.DataFrame() # Initialize as empty DataFrame on error

print(cleaned_df.shape)
print(threaded_df.shape)

Error loading cleaned_enron_emails.json: Unmatched ''"' when when decoding 'string'
Error loading threaded_emails.json: Unmatched ''"' when when decoding 'string'
(0, 0)
(0, 0)


In [ ]:
!head -n 5 /content/drive/MyDrive/Infosys_Datasets/cleaned_enron_emails.json

[
  {
    "From": "",
    "To": "",
    "Subject": "",


In [ ]:
import re
import json
import pandas as pd

file_path = "/content/drive/MyDrive/Infosys_Datasets/cleaned_enron_emails.json"

with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    raw_text = f.read()

print("Raw file loaded")
print("Total characters:", len(raw_text))

Raw file loaded
Total characters: 1048576000


In [ ]:
pattern = re.compile(r"\{.*?\}", re.DOTALL)

matches = pattern.findall(raw_text)

print("Potential records found:", len(matches))

Potential records found: 486174


In [ ]:
records = []
failed = 0

for obj in matches:
    try:
        # Fix common quote issues
        fixed = obj.replace('\n', ' ').replace('\r', ' ')
        records.append(json.loads(fixed))
    except json.JSONDecodeError:
        failed += 1

print("Recovered records:", len(records))
print("Failed records:", failed)

Recovered records: 481197
Failed records: 4977


In [ ]:
cleaned_df = pd.DataFrame(records)

print("Final DataFrame Shape:", cleaned_df.shape)
cleaned_df.head()

Final DataFrame Shape: (481197, 7)


,From,To,Subject,Date,Body,ThreadKey,Filename
0,,,,,,::,.DS_Store
1,,,,,,::,.DS_Store
2,msagel@home.com,jarnold@enron.com,Status,"Thu, 16 Nov 2000 09:30:00 -0800",John:\n?\nI'm not really sure what happened be...,"status::Thu, 16 Nov 2000 09:30:00 -0800",36.
3,slafontaine@globalp.com,john.arnold@enron.com,re:summer inverses,"Fri, 08 Dec 2000 05:05:00 -0800",i suck-hope youve made more money in natgas la...,"summer inverses::Fri, 08 Dec 2000 05:05:00 -0800",19.
4,iceoperations@intcx.com,"icehelpdesk@intcx.com, internalmarketing@intcx...",The WTI Bullet swap contracts,"Tue, 15 May 2001 09:43:00 -0700","Hi,\n\n\nFollowing the e-mail you have receive...","the wti bullet swap contracts::Tue, 15 May 200...",50.


In [ ]:
file_path = "/content/drive/MyDrive/Infosys_Datasets/threaded_emails.json"

with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
    raw_text = f.read()

matches = re.findall(r"\{.*?\}", raw_text, re.DOTALL)

records = []
failed = 0

for obj in matches:
    try:
        fixed = obj.replace('\n', ' ').replace('\r', ' ')
        records.append(json.loads(fixed))
    except json.JSONDecodeError:
        failed += 1

threaded_df = pd.DataFrame(records)

print("Threaded DF Shape:", threaded_df.shape)
print("Failed records:", failed)
threaded_df.head()

Threaded DF Shape: (235324, 11)
Failed records: 4268


,MessageID,From,To,InReplyTo,Subject,Date,Body,ThreadKey,Filename,ThreadID,ThreadPosition
0,<17938862.1075857585990.JavaMail.evans@thyme>,john.arnold@enron.com,slafontaine@globalp.com,,re:summer inverses,"Wed, 06 Dec 2000 21:38:00 -0800",seems crazy. if you're willing to ride it for...,"summer inverses::Wed, 06 Dec 2000 21:38:00 -0800",18.,thread-218b74fb-b061-43b7-a3a0-3935a88ec145,0.0
1,<23987417.1075857585124.JavaMail.evans@thyme>,slafontaine@globalp.com,john.arnold@enron.com,,re:summer inverses,"Thu, 07 Dec 2000 01:27:00 -0800",they are crazy but mite have to scale in which...,"summer inverses::Thu, 07 Dec 2000 01:27:00 -0800",23.,thread-218b74fb-b061-43b7-a3a0-3935a88ec145,3.0
2,<19171686.1075857585034.JavaMail.evans@thyme>,slafontaine@globalp.com,john.arnold@enron.com,,re:summer inverses,"Fri, 08 Dec 2000 05:05:00 -0800",i suck-hope youve made more money in natgas la...,"summer inverses::Fri, 08 Dec 2000 05:05:00 -0800",19.,thread-218b74fb-b061-43b7-a3a0-3935a88ec145,6.0
3,<3552781.1075857584209.JavaMail.evans@thyme>,john.arnold@enron.com,slafontaine@globalp.com,,re:summer inverses,"Mon, 11 Dec 2000 08:51:00 -0800",amazing how with cash futures at $1 and the ba...,"summer inverses::Mon, 11 Dec 2000 08:51:00 -0800",187.,thread-218b74fb-b061-43b7-a3a0-3935a88ec145,9.0
4,<16522398.1075857584074.JavaMail.evans@thyme>,john.arnold@enron.com,slafontaine@globalp.com,,re:summer inverses,"Mon, 11 Dec 2000 09:04:00 -0800",a couple more thoughts. certainly losing lots...,"summer inverses::Mon, 11 Dec 2000 09:04:00 -0800",181.,thread-218b74fb-b061-43b7-a3a0-3935a88ec145,13.0


In [ ]:
cleaned_df.head()
threaded_df.head()

,MessageID,From,To,InReplyTo,Subject,Date,Body,ThreadKey,Filename,ThreadID,ThreadPosition
0,<17938862.1075857585990.JavaMail.evans@thyme>,john.arnold@enron.com,slafontaine@globalp.com,,re:summer inverses,"Wed, 06 Dec 2000 21:38:00 -0800",seems crazy. if you're willing to ride it for...,"summer inverses::Wed, 06 Dec 2000 21:38:00 -0800",18.,thread-218b74fb-b061-43b7-a3a0-3935a88ec145,0.0
1,<23987417.1075857585124.JavaMail.evans@thyme>,slafontaine@globalp.com,john.arnold@enron.com,,re:summer inverses,"Thu, 07 Dec 2000 01:27:00 -0800",they are crazy but mite have to scale in which...,"summer inverses::Thu, 07 Dec 2000 01:27:00 -0800",23.,thread-218b74fb-b061-43b7-a3a0-3935a88ec145,3.0
2,<19171686.1075857585034.JavaMail.evans@thyme>,slafontaine@globalp.com,john.arnold@enron.com,,re:summer inverses,"Fri, 08 Dec 2000 05:05:00 -0800",i suck-hope youve made more money in natgas la...,"summer inverses::Fri, 08 Dec 2000 05:05:00 -0800",19.,thread-218b74fb-b061-43b7-a3a0-3935a88ec145,6.0
3,<3552781.1075857584209.JavaMail.evans@thyme>,john.arnold@enron.com,slafontaine@globalp.com,,re:summer inverses,"Mon, 11 Dec 2000 08:51:00 -0800",amazing how with cash futures at $1 and the ba...,"summer inverses::Mon, 11 Dec 2000 08:51:00 -0800",187.,thread-218b74fb-b061-43b7-a3a0-3935a88ec145,9.0
4,<16522398.1075857584074.JavaMail.evans@thyme>,john.arnold@enron.com,slafontaine@globalp.com,,re:summer inverses,"Mon, 11 Dec 2000 09:04:00 -0800",a couple more thoughts. certainly losing lots...,"summer inverses::Mon, 11 Dec 2000 09:04:00 -0800",181.,thread-218b74fb-b061-43b7-a3a0-3935a88ec145,13.0


In [ ]:
#1️⃣ DATA CLEANING

In [ ]:
#Handling Missing Values

cleaned_df.isnull().sum()

,0
From,6
To,6
Subject,6
Date,6
Body,6
ThreadKey,6
Filename,6


In [ ]:
cleaned_df = cleaned_df.dropna(subset=['Body'])

In [ ]:
cleaned_df.isnull().sum()

,0
From,0
To,0
Subject,0
Date,0
Body,0
ThreadKey,0
Filename,0


In [ ]:
#Removing Duplicate Records

before = len(cleaned_df)
cleaned_df = cleaned_df.drop_duplicates(subset=['Body']).copy() # Added .copy() to prevent SettingWithCopyWarning
after = len(cleaned_df)

print("Duplicates removed:", before - after)

Duplicates removed: 249128


In [ ]:
#Correct Invalid / Inconsistent Values

cleaned_df['Subject'] = cleaned_df['Subject'].fillna("NO_SUBJECT")

In [ ]:
cleaned_df[['Subject']].head()

,Subject
0,
2,Status
3,re:summer inverses
4,The WTI Bullet swap contracts
5,Invitation: EBS/GSS Meeting w/Bristol Babcock ...


from matplotlib import pyplot as plt
_df_0['index'].plot(kind='hist', bins=20, title='index')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
import seaborn as sns
_df_1.groupby('Subject').size().plot(kind='barh', color=sns.palettes.mpl_palette('Dark2'))
plt.gca().spines[['top', 'right',]].set_visible(False)

<string>:18: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.


from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['index']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': 'index'}, axis=1)
              .sort_values('index', ascending=True))
  xs = counted['index']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_2.sort_values('index', ascending=True)
for i, (series_name, series) in enumerate(df_sorted.groupby('Subject')):
  _plot_series(series, series_name, i)
  fig.legend(title='Subject', bbox_to_anchor=(1, 1), loc='upper left')
sns.despine(fig=fig, ax=ax)
plt.xlabel('index')
_ = plt.ylabel('count()')

from matplotlib import pyplot as plt
_df_3['index'].plot(kind='line', figsize=(8, 4), title='index')
plt.gca().spines[['top', 'right']].set_visible(False)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.



from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_4['Subject'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_4, x='index', y='Subject', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

In [ ]:
#Fixing Formatting Errors

cleaned_df['Erom'] = cleaned_df['From'].str.lower().str.strip()
cleaned_df['To'] = cleaned_df['To'].str.lower().str.strip()

In [ ]:
cleaned_df[['From', 'To']].head()

,From,To
0,,
2,msagel@home.com,jarnold@enron.com
3,slafontaine@globalp.com,john.arnold@enron.com
4,iceoperations@intcx.com,"icehelpdesk@intcx.com, internalmarketing@intcx..."
5,jeff.youngflesh@enron.com,"anthony.gilmore@enron.com, colleen.koenig@enro..."


In [ ]:
#2️⃣ DATA VALIDATION

In [ ]:
#Schema Validation

expected_columns = ['From', 'To', 'Subject', 'Body', 'Date']
missing_cols = set(expected_columns) - set(cleaned_df.columns)

print("Missing columns:", missing_cols)

Missing columns: set()


In [ ]:
#Mandatory Field Checks

mandatory_check = cleaned_df[['From', 'To', 'Body']].isnull().sum()
mandatory_check

,0
From,0
To,0
Body,0


In [ ]:
#Range / Logical Checks

invalid_dates = cleaned_df['Date'].isnull().sum()
print("Invalid Dates:", invalid_dates)

Invalid Dates: 0


In [ ]:
#Referential Integrity (Threads)

threaded_df[['ThreadID']].head()

,ThreadID
0,thread-218b74fb-b061-43b7-a3a0-3935a88ec145
1,thread-218b74fb-b061-43b7-a3a0-3935a88ec145
2,thread-218b74fb-b061-43b7-a3a0-3935a88ec145
3,thread-218b74fb-b061-43b7-a3a0-3935a88ec145
4,thread-218b74fb-b061-43b7-a3a0-3935a88ec145


In [ ]:
#3️⃣ DATA TRANSFORMATION

In [ ]:
#Data Type Conversion

cleaned_df['Date'] = pd.to_datetime(cleaned_df['Date'], errors='coerce', utc=True)

In [ ]:
cleaned_df.dtypes

,0
From,object
To,object
Subject,object
Date,"datetime64[ns, UTC]"
Body,object
ThreadKey,object
Filename,object
Erom,object


In [ ]:
#Unit Conversion (Timezone → UTC)

cleaned_df['Date_utc'] = cleaned_df['Date'].dt.tz_convert('UTC')
cleaned_df[['Date_utc']].head()

,Date_utc
0,NaT
2,2000-11-16 17:30:00+00:00
3,2000-12-08 13:05:00+00:00
4,2001-05-15 16:43:00+00:00
5,2000-11-27 09:49:00+00:00


In [ ]:
#Normalization

cleaned_df['email_length'] = cleaned_df['Body'].apply(len)

scaler = MinMaxScaler()
cleaned_df['email_length_norm'] = scaler.fit_transform(
    cleaned_df[['email_length']]
)

cleaned_df[['email_length', 'email_length_norm']].head()

,email_length,email_length_norm
0,0,0.000000
2,600,0.000298
3,275,0.000137
4,1066,0.000530
5,198,0.000098


In [ ]:
#Aggregation (Daily → Monthly)

cleaned_df['month'] = cleaned_df['Date_utc'].dt.to_period('M')

monthly_counts = cleaned_df.groupby('month').size().reset_index(name='email_count')
monthly_counts.head()

/tmp/ipython-input-939154425.py:3: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  cleaned_df['month'] = cleaned_df['Date_utc'].dt.to_period('M')


,month,email_count
0,1980-01,273
1,1986-04,1
2,1986-05,1
3,1997-01,1
4,1997-03,10


In [ ]:
#Encoding (Categorical → Numeric)

encoder = LabelEncoder()
cleaned_df['sender_id'] = encoder.fit_transform(cleaned_df['From'])

cleaned_df[['From', 'sender_id']].head()

,From,sender_id
0,,0
2,msagel@home.com,11714
3,slafontaine@globalp.com,15187
4,iceoperations@intcx.com,6795
5,jeff.youngflesh@enron.com,7576


In [ ]:
#4️⃣ DATA FILTERING

In [ ]:
#1 Remove Unnecessary Columns

filtered_df = cleaned_df[['From', 'To', 'Subject', 'Body', 'Date_utc']].copy()
filtered_df.head()

,From,To,Subject,Body,Date_utc
0,,,,,NaT
2,msagel@home.com,jarnold@enron.com,Status,John:\n?\nI'm not really sure what happened be...,2000-11-16 17:30:00+00:00
3,slafontaine@globalp.com,john.arnold@enron.com,re:summer inverses,i suck-hope youve made more money in natgas la...,2000-12-08 13:05:00+00:00
4,iceoperations@intcx.com,"icehelpdesk@intcx.com, internalmarketing@intcx...",The WTI Bullet swap contracts,"Hi,\n\n\nFollowing the e-mail you have receive...",2001-05-15 16:43:00+00:00
5,jeff.youngflesh@enron.com,"anthony.gilmore@enron.com, colleen.koenig@enro...",Invitation: EBS/GSS Meeting w/Bristol Babcock ...,Conference Room TBD.\n\nThis meeting will be t...,2000-11-27 09:49:00+00:00


In [ ]:
#2 Drop Incomplete Records

filtered_df = filtered_df.dropna()
filtered_df.shape

(232061, 5)

In [ ]:
#5️⃣ DATA ENRICHMENT

In [ ]:
#1 Derived Fields

filtered_df['word_count'] = filtered_df['Body'].apply(lambda x: len(x.split()))
filtered_df[['word_count']].head()

,word_count
2,104
3,51
4,170
5,29
6,137


In [ ]:
#2 Domain Extraction (Enrichment)

filtered_df['sender_domain'] = filtered_df['From'].apply(lambda x: x.split('@')[-1])
filtered_df[['From', 'sender_domain']].head()

,From,sender_domain
2,msagel@home.com,home.com
3,slafontaine@globalp.com,globalp.com
4,iceoperations@intcx.com,intcx.com
5,jeff.youngflesh@enron.com,enron.com
6,caroline.abramo@enron.com,enron.com


In [ ]:
#6️⃣ DATA DEDUPLICATION

In [ ]:
#1 Exact Match Deduplication

before = len(filtered_df)
filtered_df = filtered_df.drop_duplicates()
after = len(filtered_df)

print("Deduplicated:", before - after)

Deduplicated: 0


In [ ]:
#2 Primary-Key Deduplication

filtered_df = filtered_df.drop_duplicates(subset=['From', 'Date_utc'])
filtered_df.head()

,From,To,Subject,Body,Date_utc,word_count,sender_domain
2,msagel@home.com,jarnold@enron.com,Status,John:\n?\nI'm not really sure what happened be...,2000-11-16 17:30:00+00:00,104,home.com
3,slafontaine@globalp.com,john.arnold@enron.com,re:summer inverses,i suck-hope youve made more money in natgas la...,2000-12-08 13:05:00+00:00,51,globalp.com
4,iceoperations@intcx.com,"icehelpdesk@intcx.com, internalmarketing@intcx...",The WTI Bullet swap contracts,"Hi,\n\n\nFollowing the e-mail you have receive...",2001-05-15 16:43:00+00:00,170,intcx.com
5,jeff.youngflesh@enron.com,"anthony.gilmore@enron.com, colleen.koenig@enro...",Invitation: EBS/GSS Meeting w/Bristol Babcock ...,Conference Room TBD.\n\nThis meeting will be t...,2000-11-27 09:49:00+00:00,29,enron.com
6,caroline.abramo@enron.com,mike.grigsby@enron.com,Harvard Mgmt,Mike- I have their trader coming into the offi...,2000-12-12 17:33:00+00:00,137,enron.com


In [ ]:
#7️⃣ DATA MASKING & SECURITY

In [ ]:
#1 Mask Email Addresses

def mask_email(email):
    return email[:2] + "****@" + email.split('@')[-1]

filtered_df['from_masked'] = filtered_df['From'].apply(mask_email)
filtered_df[['From', 'from_masked']].head()

,From,from_masked
2,msagel@home.com,ms****@home.com
3,slafontaine@globalp.com,sl****@globalp.com
4,iceoperations@intcx.com,ic****@intcx.com
5,jeff.youngflesh@enron.com,je****@enron.com
6,caroline.abramo@enron.com,ca****@enron.com


In [ ]:
#2 Hashing

import hashlib

filtered_df['from_hash'] = filtered_df['From'].apply(
    lambda x: hashlib.sha256(x.encode()).hexdigest()
)

filtered_df[['from_hash']].head()

,from_hash
2,17f17dd4074f3955606b951741ea785f2360278b50488a...
3,b35c238d5a30faf4834ebd80d93a72b8e365795c57493e...
4,b633b679b3fe3255dddd3f6589837bc4a4d7e717f64fd7...
5,4cc1213fff277a44aebb500d15668e0876e7cd698b8008...
6,86f6e878c74072f2726a20be9e94a22f57e51161491701...


In [ ]:
#8️⃣ DATA STANDARDIZATION

In [ ]:
filtered_df.columns = [c.lower().replace(" ", "_") for c in filtered_df.columns]
filtered_df.head()

,from,to,subject,body,date_utc,word_count,sender_domain,from_masked,from_hash
2,msagel@home.com,jarnold@enron.com,Status,John:\n?\nI'm not really sure what happened be...,2000-11-16 17:30:00+00:00,104,home.com,ms****@home.com,17f17dd4074f3955606b951741ea785f2360278b50488a...
3,slafontaine@globalp.com,john.arnold@enron.com,re:summer inverses,i suck-hope youve made more money in natgas la...,2000-12-08 13:05:00+00:00,51,globalp.com,sl****@globalp.com,b35c238d5a30faf4834ebd80d93a72b8e365795c57493e...
4,iceoperations@intcx.com,"icehelpdesk@intcx.com, internalmarketing@intcx...",The WTI Bullet swap contracts,"Hi,\n\n\nFollowing the e-mail you have receive...",2001-05-15 16:43:00+00:00,170,intcx.com,ic****@intcx.com,b633b679b3fe3255dddd3f6589837bc4a4d7e717f64fd7...
5,jeff.youngflesh@enron.com,"anthony.gilmore@enron.com, colleen.koenig@enro...",Invitation: EBS/GSS Meeting w/Bristol Babcock ...,Conference Room TBD.\n\nThis meeting will be t...,2000-11-27 09:49:00+00:00,29,enron.com,je****@enron.com,4cc1213fff277a44aebb500d15668e0876e7cd698b8008...
6,caroline.abramo@enron.com,mike.grigsby@enron.com,Harvard Mgmt,Mike- I have their trader coming into the offi...,2000-12-12 17:33:00+00:00,137,enron.com,ca****@enron.com,86f6e878c74072f2726a20be9e94a22f57e51161491701...


In [ ]:
#9️⃣ ERROR HANDLING & LOGGING

In [ ]:
error_log = []

try:
    pd.to_datetime("invalid-date")
except Exception as e:
    error_log.append(str(e))

error_log

['Unknown datetime string format, unable to parse: invalid-date, at position 0']

In [ ]:
error_log = []

def safe_to_datetime(value, column_name="unknown"):
    try:
        return pd.to_datetime(value)
    except Exception as e:
        error_log.append({
            "column": column_name,
            "value": value,
            "error": str(e)
        })
        return pd.NaT

In [ ]:
#🔟 METADATA HANDLING

In [ ]:
filtered_df['ingestion_time'] = datetime.utcnow()
filtered_df['source_system'] = 'Enron Email Dataset'
filtered_df['data_version'] = 'v1.0'

filtered_df[['ingestion_time', 'source_system', 'data_version']].head()

/tmp/ipython-input-3829850487.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  filtered_df['ingestion_time'] = datetime.utcnow()


,ingestion_time,source_system,data_version
2,2026-01-21 06:40:52.049086,Enron Email Dataset,v1.0
3,2026-01-21 06:40:52.049086,Enron Email Dataset,v1.0
4,2026-01-21 06:40:52.049086,Enron Email Dataset,v1.0
5,2026-01-21 06:40:52.049086,Enron Email Dataset,v1.0
6,2026-01-21 06:40:52.049086,Enron Email Dataset,v1.0


In [ ]:
#1️⃣1️⃣ SAMPLING

In [ ]:
sample_df = filtered_df.sample(n=5, random_state=42)
sample_df

,from,to,subject,body,date_utc,word_count,sender_domain,from_masked,from_hash,ingestion_time,source_system,data_version
159440,jeff.dasovich@enron.com,mday@gmssr.com,Rosenfield exposes phony bankruptcy,----- Forwarded by Jeff Dasovich/NA/Enron on 1...,2000-12-20 12:56:00+00:00,212,enron.com,je****@enron.com,5a37ed79e09fa97afff4b9f6e3126e907a3cf6cddae9dd...,2026-01-21 06:40:52.049086,Enron Email Dataset,v1.0
6302,maggie_timmins@cdnoxy.com,"afowler@arcfinancial.com, alkeo@hotmail.com, a...",WHERE ARE THE BALLOON PRIZES?,"Okay fellow deputies, where are your prizes?\n...",2000-01-13 19:45:00+00:00,223,cdnoxy.com,ma****@cdnoxy.com,24c91fb28058b7df58af14d3af059b033ad3db1f1a6407...,2026-01-21 06:40:52.049086,Enron Email Dataset,v1.0
62330,mike.mcconnell@enron.com,ken.rice@enron.com,Re:,Thanks for the note. Sooners rule. We'll fi...,2001-01-19 07:55:00+00:00,151,enron.com,mi****@enron.com,b61bb4d0bb6d25335d469f9a6efcad0508c9143fc0ae4c...,2026-01-21 06:40:52.049086,Enron Email Dataset,v1.0
398667,mark.haedicke@enron.com,mcunningham@isda.org,Re: Pacific Forest Resources,"At this point, I think they should be a subscr...",2000-03-09 17:52:00+00:00,20,enron.com,ma****@enron.com,04b35390ad540dff884609d0be32537a92bc60f65502e4...,2026-01-21 06:40:52.049086,Enron Email Dataset,v1.0
276591,lisa.bills@enron.com,kay.mann@enron.com,Re: Winston & Strawn Comment: Consents to Assi...,"Kay, I have told Rob that we are not pushing G...",2000-12-10 15:33:00+00:00,368,enron.com,li****@enron.com,03484b6b18b098bb27d694d743863b83d8de8f33b94776...,2026-01-21 06:40:52.049086,Enron Email Dataset,v1.0


In [ ]:
filtered_df.to_csv('final_ingested_enron_emails.csv', index=False)
monthly_counts.to_csv('monthly_email_aggregation.csv', index=False)

In [ ]:
text_col = "final_text"

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 43.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

def extract_entities(text):
    doc = nlp(text)
    return [(ent.text, ent.label_) for ent in doc.ents]

df["entities"] = df[text_col].apply(extract_entities)

NameError: name 'df' is not defined

In [ ]:
df.head()
df.columns

NameError: name 'df' is not defined

In [ ]:
print("df" in globals())

False


In [ ]:
import pandas as pd

df = pd.read_json("/content/drive/MyDrive/Infosys_Datasets/cleaned_enron_emails.json", lines=True)

ValueError: Expected object or value

In [ ]:
import os
os.path.getsize("/content/drive/MyDrive/Infosys_Datasets/cleaned_enron_emails.json")

1048576000

In [ ]:
{"id":1,"text":"hello"}
{"id":2,"text":"world"}

{'id': 2, 'text': 'world'}

In [ ]:
import pandas as pd
df = pd.read_json("/content/drive/MyDrive/Infosys_Datasets/cleaned_enron_emails.json")

ValueError: Unmatched ''"' when when decoding 'string'

In [ ]:
with open("/content/drive/MyDrive/Infosys_Datasets/cleaned_enron_emails.json", errors="ignore") as f:
    for i in range(10):
        print(f.readline())

[

  {

    "From": "",

    "To": "",

    "Subject": "",

    "Date": "",

    "Body": "",

    "ThreadKey": "::",

    "Filename": ".DS_Store"

  },



In [ ]:
import json

with open("/content/drive/MyDrive/Infosys_Datasets/cleaned_enron_emails.json", errors="ignore") as f:
    raw = f.read()

data = json.loads(raw)

JSONDecodeError: Unterminated string starting at: line 4343002 column 13 (char 1048499281)

In [ ]:
with open("/content/drive/MyDrive/Infosys_Datasets/cleaned_enron_emails.json", errors="ignore") as f:
    for i, line in enumerate(f):
        if i == 4343001:   # zero-based index
            print(line)
            break

    "Body": "Please see the following articles:\n\nSac Bee, Fri, 6/22: Employees: Power supply held down\n\nSac Bee, Fri, 6/22: Consumers cut down their own power in protest\n\nSac Bee, Fri, 6/22: Davis consultants had contract with Edison: The disclos=\nures turn up the heat on the governor for hiring=20\nex-Clinton aides\n\nSD Union, Fri, 6/22: State deal may ease blackout threat\nCanada to supply energy as summer demand rises=20\n\nSD Union, Fri, 6/22: Ex-worker: Duke manipulated market\n\nLA Times, Fri, 6/22: Estimates of power profits disputed\n\nLA Times, Fri, 6/22: Edison plans bond offer at 13% rate\n\nLA Times, Fri, 6/22: Energy company abandons plans for Baldwin Hills plant=\n=20\n\nSF Chron, Fri, 6/22: Western states could feel pinch from California pricin=\ng=20\n\nSF Chron, Fri, 6/22: Feds spurn Duke Energy in its bid to avoid refunds\n\nSF Chron, Fri, 6/22: News Analysis: Davis winning Washington PR battle=20\nPrice cap victory may rob Democrats of campaign issue\n\nSF Ch

In [ ]:
import json
import pandas as pd

good_rows = []

with open("/content/drive/MyDrive/Infosys_Datasets/cleaned_enron_emails.json", errors="ignore") as f:
    for line in f:
        try:
            good_rows.append(json.loads(line))
        except:
            continue   # skip broken rows

df = pd.DataFrame(good_rows)

In [ ]:
df.shape
df.head()
df.columns

RangeIndex(start=0, stop=0, step=1)

In [ ]:
df.shape

(0, 0)

In [ ]:
df.head()


""


from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['index']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': 'index'}, axis=1)
              .sort_values('index', ascending=True))
  xs = counted['index']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_0.sort_values('index', ascending=True)
_plot_series(df_sorted, '')
sns.despine(fig=fig, ax=ax)
plt.xlabel('index')
_ = plt.ylabel('count()')

In [ ]:
df.size

0